In [17]:
# STEP 1: Install dependencies
!pip install openai pandas

In [18]:
# STEP 2: Imports
import pandas as pd
import json
from openai import OpenAI

In [19]:
from httpx import Client
# STEP 3: Setup API
import os

os.environ["OPENAI_API_KEY"] = "Enter_Your_API_KEY"


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [20]:
from google.colab import files
uploaded = files.upload()

Saving Job_descriptions.csv to Job_descriptions (2).csv


In [21]:
# Get filename
filename = list(uploaded.keys())[0]

In [22]:

# Read CSV
df_input = pd.read_csv(filename)

# Extract job descriptions column
job_descriptions = df_input['Job_descriptions'].dropna().tolist()

print(f"✅ Loaded {len(job_descriptions)} job descriptions")

✅ Loaded 25 job descriptions


In [23]:
# STEP 5: EXTRACTION FUNCTION
def extract_job_data(text):
    prompt = f"""
    Extract structured information from the job description.

    Return ONLY valid JSON with:
    - role
    - seniority
    - experience
    - skills (list)
    - location
    - salary

    If any field is missing, return "Not specified".

    Job Description:
    {text}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [24]:
# STEP 6: RUN PIPELINE
structured_data = []

for i, jd in enumerate(job_descriptions):
    try:
        result = extract_job_data(jd)

        # Clean response (important for JSON parsing)
        result = result.strip().replace("```json", "").replace("```", "")

        parsed = json.loads(result)
        structured_data.append(parsed)

        print(f"Processed {i+1}/{len(job_descriptions)}")

    except Exception as e:
        print(f"❌ Error at {i}: {e}")

        structured_data.append({
            "role": "Error",
            "seniority": "Error",
            "experience": "Error",
            "skills": [],
            "location": "Error",
            "salary": "Error"
        })

Processed 1/25
Processed 2/25
Processed 3/25
Processed 4/25
Processed 5/25
Processed 6/25
Processed 7/25
Processed 8/25
Processed 9/25
Processed 10/25
Processed 11/25
Processed 12/25
Processed 13/25
Processed 14/25
Processed 15/25
Processed 16/25
Processed 17/25
Processed 18/25
Processed 19/25
Processed 20/25
Processed 21/25
Processed 22/25
Processed 23/25
Processed 24/25
Processed 25/25


In [25]:
# STEP 7: CREATE DATAFRAME
df_output = pd.DataFrame(structured_data)

# STEP 8: SAVE OUTPUT CSV
df_output.to_csv("structured_jobs.csv", index=False)

print("\n✅ CSV file 'structured_jobs.csv' created successfully!")


✅ CSV file 'structured_jobs.csv' created successfully!


In [26]:
# STEP 9: BONUS — SAVE RAW + STRUCTURED TOGETHER (VERY IMPRESSIVE)
df_final = pd.DataFrame({
    "raw_job_description": job_descriptions,
    "structured_output": structured_data
})

df_final.to_csv("final_output_with_raw.csv", index=False)

# Preview
df_output.head()

,role,seniority,experience,skills,location,salary
0,Python Developer Trainee,Entry-level,0 - 2 years,"[Python scripting, SQL knowledge, Requirement ...",Ahmedabad,4-5 Lacs P.A.
1,Computer Vision ML Engineer,Not specified,0 - 2 years,"[YOLO, OpenCV, PyTorch, Transformers, Python, ...",Bengaluru,Not specified
2,Data Scientist,Not specified,0 - 6 years,"[data analysis, machine learning, statistics, ...",Gurugram,Not specified
3,Data Scientist,Senior,5+ years,"[Python, NLP libraries, Hugging Face Transform...",Ahmedabad( Gota ),Not specified
4,Data Scientist,Not specified,0 - 3 years,"[machine learning, data analysis, predictive m...",Bengaluru,Not specified
